# Step 3 Part A: Black-Scholes delta hedging on a single option (prototype)

Important: our data is one real day per month. Each option's hedging history exists only WITHIN a single sampled day (~24 hourly steps), not continuously across weeks. So we prototype the simulation loop on ONE option, ON ONE sampled day.

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("btc_options_train.csv")
train["hour_bucket"] = pd.to_datetime(train["hour_bucket"])
train["sample_date"] = train["hour_bucket"].dt.date
print(f"{len(train)} rows, {train['symbol'].nunique()} unique instruments, {train['sample_date'].nunique()} sampled days")

## Find a good prototype: one (symbol, sample_date) pair with many hourly rows

In [ ]:
counts = train.groupby(["symbol", "sample_date"]).size().sort_values(ascending=False)
counts.head(10)

In [ ]:
# Pick the top one, or swap in a specific (symbol, sample_date) pair yourself
proto_symbol, proto_date = counts.index[0]
print(f"Prototyping on: {proto_symbol} on {proto_date}")

episode = train[(train["symbol"] == proto_symbol) & (train["sample_date"] == proto_date)].sort_values("hour_bucket").reset_index(drop=True)
episode[["hour_bucket", "type", "strike_price", "underlying_price", "time_to_maturity_days", "mark_iv", "bid_price", "ask_price", "mid_price"]]

## Black-Scholes delta
Using observed `mark_iv` directly -- no need to invert BS ourselves since Deribit already computed it.

In [ ]:
from scipy.stats import norm

def bs_delta(S, K, T_years, sigma, option_type, r=0.0):
    """Black-Scholes delta. S=spot, K=strike, T_years=time to maturity in years, sigma=annualized vol (decimal), option_type='call'/'put'."""
    if T_years <= 0 or sigma <= 0:
        # at/after expiry, delta is just the payoff indicator
        if option_type == "call":
            return 1.0 if S > K else 0.0
        else:
            return -1.0 if S < K else 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T_years) / (sigma * np.sqrt(T_years))
    if option_type == "call":
        return norm.cdf(d1)
    else:
        return norm.cdf(d1) - 1.0

episode["T_years"] = episode["time_to_maturity_days"] / 365
episode["iv_decimal"] = episode["mark_iv"] / 100  # mark_iv looked like it's in percent (e.g. 52.7), check this
episode["bs_delta"] = episode.apply(
    lambda row: bs_delta(row["underlying_price"], row["strike_price"], row["T_years"], row["iv_decimal"], row["type"]),
    axis=1
)
episode[["hour_bucket", "underlying_price", "strike_price", "iv_decimal", "bs_delta"]]

## Sanity check: plot delta over the day
Does it move sensibly as spot price moves relative to strike?

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(episode["hour_bucket"], episode["underlying_price"], marker="o")
axes[0].axhline(episode["strike_price"].iloc[0], color="gray", linestyle="--", label="strike")
axes[0].set_title(f"{proto_symbol}: underlying price vs strike")
axes[0].legend()

axes[1].plot(episode["hour_bucket"], episode["bs_delta"], marker="o", color="darkgreen")
axes[1].set_title("Black-Scholes delta")
axes[1].set_ylim(-1.1, 1.1)
plt.tight_layout()
plt.show()